In [1]:
import torch
from typing import Tuple

import cutlass
import cutlass.cute as cute
import cutlass.utils as utils
from cutlass.cute.runtime import from_dlpack
from cutlass.cute.testing import benchmark, JitArguments

In [4]:


class Test:
    def __init__(
    self,
    cta_tiler: Tuple[int, int, int] = (16, 16, 16)
    ):
        self.cta_tiler = cta_tiler
        self.BM, self.BN, self.BK = cta_tiler
    @cute.jit
    def __call__(
        self,
        Q: cute.Tensor,
    ):
        self.kernel(
            Q
        ).launch(
            grid=(Q.shape[0]//self.BM, Q.shape[1]//self.BN, 1),
            block=(self.BM, self.BN, 1)
        )

    @cute.kernel
    def kernel(
        self,
        Q: cute.Tensor,
    ):  
        bidx, bidy, _ = cute.arch.block_idx()
        gQ_mk = cute.zipped_divide(Q, (self.BM, 1))
        print("gQ_mk: ", gQ_mk)
        
        gQ = gQ_mk[(None, None), (bidx, None)]
        print("gQ: ", gQ)
        
        gQ_ = cute.zipped_divide(Q, (self.BM, self.BN))
        print("gQ_: ", gQ_)
        gQ_out = gQ_[(None, None), (bidx, bidy)]
        print("gQ_out: ", gQ_out)
        
        

kernel = Test()

M, K = 1024, 1024
Q = torch.randn((M, K), device="cuda", dtype=torch.float16)

kernel(from_dlpack(Q, assumed_align=16))

gQ_mk:  tensor<ptr<f16, gmem, align<16>> o ((16,1),(64,1024)):((1024,0),(16384,1))>
gQ:  tensor<ptr<f16, gmem, align<16>> o (16,1,1024):(1024,0,1)>
gQ_:  tensor<ptr<f16, gmem, align<16>> o ((16,16),(64,64)):((1024,1),(16384,16))>
gQ_out:  tensor<ptr<f16, gmem, align<16>> o (16,16):(1024,1)>
